In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import math

In [ ]:
def conv(image, filter):
    h, w = image.shape
    k_h = int(len(filter)/2)
    k_w = int(len(filter[0])/2)
    
    # initialize image after conv
    image_conv = np.zeros((h, w))

    # padding
    padded_image = np.pad(image, ((k_h, k_h), (k_w, k_w)), mode='edge')
    
    # convolving
    for i in range(h):
        for j in range(w):
            image_conv[i][j] = np.einsum("ij, ij -> ", padded_image[i:i+(2*k_h+1), j:j+(2*k_w+1)], filter[::-1,::-1])
    
    return image_conv

def edge_detection(image, sigma):
    # gaussian filter
    def gaussian_filter(k_h, k_w, sigma):
        gaussian = np.zeros((2*k_h+1, 2*k_w+1))
        for i in range(0, 2*k_h+1):
            for j in range(0, 2*k_w+1):
                gaussian[i][j] = math.exp(-((k_h-i)**2 + (k_w-j)**2)/(2 * sigma**2))/(2 * math.pi * sigma**2)

        return gaussian

    # sobel filter
    def sobel_filter(mode):
        if mode=='X':
            return np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
        if mode=='Y':
            return np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])

    # Non Maximal Suppresion
    def NMS(I_m, I_o):
        h, w = I_m.shape

        I_suppresed = np.zeros((h, w))
        for i in range(h):
            for j in range(w):
                magnitude = I_m[i][j]
                orientation = I_o[i][j]

                is_edge = True
                if orientation >= - 1/8 * np.pi and orientation < 1/8 * np.pi:
                    if (i-1 >=0 and I_m[i-1][j] > magnitude) or (i+1 <= h-1 and I_m[i+1][j] > magnitude):
                        is_edge = False
                elif orientation >= 1/8 * np.pi and orientation < 3/8 * np.pi:
                    if (i-1 >=0 and j+1 <= w-1 and I_m[i-1][j+1] > magnitude) or (i+1 <= h-1 and j-1 >=0 and I_m[i+1][j-1] > magnitude):
                        is_edge = False
                elif orientation >= -3/8 * np.pi and orientation < -1/8 * np.pi:
                    if (i-1 >=0 and j-1 >= 0 and I_m[i-1][j-1] > magnitude) or (i+1 <= h-1 and j+1 <= w-1 and I_m[i+1][j+1] > magnitude):
                        is_edge = False
                else:
                    if (j-1 >= 0 and I_m[i][j-1] > magnitude) or (j+1 <= w-1 and I_m[i][j+1] > magnitude):
                        is_edge = False
                
                if is_edge:
                    I_suppresed[i][j] = I_m[i][j]
                else:
                    I_suppresed[i][j] = 0
        
        return I_suppresed
        
    # gaussian blurring
    gaussian = gaussian_filter(1, 1, sigma)
    image_blurred = conv(image, filter=gaussian)    

    # gradient by X, Y
    I_x = conv(image_blurred, filter=sobel_filter(mode='X'))
    I_y = conv(image_blurred, filter=sobel_filter(mode='Y'))

    # avoid zero division
    I_y[I_y == 0] += 1e-6

    # Edge Magnitude, Orientation
    I_m = np.sqrt(I_x ** 2 + I_y ** 2)
    I_o = np.arctan(I_x / I_y)

    # Non Maximal Suppression
    I_m = NMS(I_m, I_o)

    return I_m, I_o, I_x, I_y

In [ ]:
def hough_lines(image, threshold, rho_res, theta_res):
    
    h, w = image.shape
    diagonal = np.sqrt(h**2 + w**2)

    thetas = np.arange(0, np.pi, theta_res)
    rhos_num = int(2 * diagonal / rho_res) + 1

    H = np.zeros((rhos_num, len(thetas)))  # 튜플로 shape 지정

    y_edges, x_edges = np.where(image >= threshold)

    # 미리 계산
    sin_theta = np.sin(thetas)
    cos_theta = np.cos(thetas)

    for x, y in zip(x_edges, y_edges):

        rho = x * cos_theta + y * sin_theta

        rho_index = np.round((rho + diagonal) / rho_res).astype(int)

        ok_index = (rho_index >= 0) & (rho_index < rhos_num)

        H[rho_index[ok_index], np.where(ok_index)[0]] += 1
    
    return H

In [ ]:
def hough_lines_peak(H, rho_res, theta_res, num_lines):

    rhos_num = H.shape[0]
    diagonal = (rhos_num - 1) * rho_res / 2

    temp_H = H.copy().astype(float)

    line_rho_list = []
    line_theta_list = []

    rho_suppression = max(int(20 / rho_res), 1)
    theta_suppression = max(int(np.radians(10) / theta_res), 1)

    for _ in range(num_lines):

        # 가장 많은 투표 위치 선택
        index = np.argmax(temp_H)

        rho_index, theta_index = np.unravel_index(index, temp_H.shape)

        rho = rho_res * rho_index - diagonal
        line_rho_list.append(rho)

        theta = theta_res * theta_index
        line_theta_list.append(theta)

        # NMS

        rho_low = max(rho_index - rho_suppression, 0)

        theta_low = max(theta_index - theta_suppression, 0)

        rho_high = min(1 + rho_index + rho_suppression, temp_H.shape[0])

        theta_high = min(1 + theta_index + theta_suppression, temp_H.shape[1])

        temp_H[rho_low : rho_high, theta_low : theta_high] = 0

    return line_rho_list, line_theta_list

In [ ]:
def hough_circles(image, threshold, radius_range, a_res, b_res, r_res):

    y_edge, x_edge = np.where(image >= threshold)

    ##################################

    h, w = image.shape
    min_r, max_r = radius_range

    num_a = 1 + int(w / a_res)
    num_b = 1 + int(h / b_res)
    num_r = 1 + int((max_r - min_r) / r_res)

    H = np.zeros((num_b, num_a, num_r), dtype = np.int64)

    #################################

    angles = np.arange(0, 2 * np.pi, np.pi / 90)

    cos_a = np.cos(angles)
    sin_a = np.sin(angles)

    r_list = min_r + np.arange(num_r) * r_res

    for x, y in zip(x_edge, y_edge):

        for r_index, r in enumerate(r_list):

            a = np.round((x - r * cos_a) / a_res).astype(int)
            b = np.round((y - r * sin_a) / b_res).astype(int)

            ok = (a >= 0) & (a < num_a) & (b >= 0) & (b < num_b)

            np.add.at(H[:, :, r_index], (b[ok], a[ok]), 1)


    return H

In [ ]:
def hough_circles_peak(H, radius_range, a_res, b_res, r_res, num_circles):
    
    circle_a = []
    circle_b = []
    circle_c = []

    a_suppression = max(int(20 / a_res), 1)
    b_suppression = max(int(20 / b_res), 1)
    r_suppression = max(int(20 / r_res), 1)

    temp_H = H.copy().astype(float)

    min_r = radius_range[0]

    for _ in range(num_circles):

        index = np.argmax(temp_H)

        b_index, a_index, r_index = np.unravel_index(index, temp_H.shape)

        circle_a.append(a_index * a_res)
        circle_b.append(b_index * b_res)
        circle_c.append(min_r + r_index * r_res)

        a_low = max(a_index - a_suppression, 0)
        b_low = max(b_index - b_suppression, 0)
        r_low = max(r_index - r_suppression, 0)

        a_high = min(1 + a_index + a_suppression, temp_H.shape[1]) 
        b_high = min(1 + b_index + b_suppression, temp_H.shape[0])  
        r_high = min(1 + r_index + r_suppression, temp_H.shape[2])  

        temp_H[b_low : b_high, a_low : a_high, r_low : r_high] = 0

    return circle_a, circle_b, circle_c

In [ ]:
def draw_lines(image, line_rhos, line_thetas):
    w, h = image.size
    image_with_lines = image.copy()
    image_draw = ImageDraw.Draw(image_with_lines)  

    for i in range(len(line_rhos)):
        rho = line_rhos[i]
        theta = line_thetas[i]
    
        a = np.cos(theta)
        b = np.sin(theta)

        # Avoid zero division
        if b == 0: b += 1e-6

        m = - a / b
        c = rho / b

        if abs(m) < 1:
            x1 = 0 
            y1 = m * x1 + c
            x2 = w - 1
            y2 = m * x2 + c

            image_draw.line([(x1, y1), (x2, y2)], fill='orange', width=3)
        else:
            y1 = 0
            x1 = (y1 - c) / m
            y2 = h - 1
            x2 = (y2 - c) / m

            image_draw.line([(x1, y1), (x2, y2)], fill='orange', width=3)
    
    return image_with_lines


def draw_circles(image, circle_as, circle_bs, circle_rs):
    image_with_circles = image.copy()
    image_draw = ImageDraw.Draw(image_with_circles)  

    for a, b, r in zip(circle_as, circle_bs, circle_rs):
        image_draw.ellipse((a-r, b-r, a+r, b+r), outline='orange', fill=None, width=3)
    
    return image_with_circles

In [ ]:
######## Parameters ########
# feel free to change this
sigma=2
threshold=0.08
rho_res=2.3
theta_res=math.pi/180
num_lines=20
######## Parameters ########

image = Image.open("./00000.png")
image_grey = np.array(image.convert("L"))
image_grey = image_grey / 255.

edge_image, _, __, ___ = edge_detection(image_grey, sigma)
H = hough_lines(edge_image, threshold, rho_res, theta_res)
line_rhos, line_thetas = hough_lines_peak(H, rho_res, theta_res, num_lines)
image_with_lines = draw_lines(image, line_rhos, line_thetas)

image_with_lines.save("./00000_output.png")

In [ ]:
######## Parameters ########
# feel free to change this
sigma=2
threshold=0.1
a_res=2.0
b_res=2.0
r_res=4.0
radius_range=(120, 180)
num_circles=3
######## Parameters ########

image = Image.open("./00001.png")
image_grey = np.array(image.convert("L"))
image_grey = image_grey / 255.

edge_image, _, __, ___ = edge_detection(image_grey, sigma)
H = hough_circles(edge_image, threshold, radius_range, a_res, b_res, r_res)
circle_as, circle_bs, circle_rs = hough_circles_peak(H, radius_range, a_res, b_res, r_res, num_circles)
image_with_circles = draw_circles(image, circle_as, circle_bs, circle_rs)

image_with_circles.save("./00001_output.png")

## My image Test

In [33]:
import os
from itertools import product

######## Parameters ########
# default #
# sigma=2 # 얼마나 흐릿하게 만드는가? 커질수록 노이즈 제거 강화
# threshold=0.08 # 최소 강도
# rho_res=2.3 # 정밀도; 작을수록 정밀
# theta_res=math.pi/180
# num_lines=20 # 검출할 직선 개수

sigma_list     = [0.5, 1, 2, 3, 5]
threshold_list = [0.02, 0.04, 0.08, 0.1, 0.2]
rho_res_list   = [1.0, 1.5, 2.3, 3.0]
num_lines_list = [15, 20, 30]
theta_res      = math.pi / 180
######## Parameters ########

image = Image.open("./my_image_01.jpg")
image_grey = np.array(image.convert("L")) / 255.

total = len(sigma_list) * len(threshold_list) * len(rho_res_list) * len(num_lines_list)
count = 0

for sigma in sigma_list:

    edge_image, _, __, ___ = edge_detection(image_grey, sigma)

    for threshold, rho_res in product(threshold_list, rho_res_list):
        H = hough_lines(edge_image, threshold, rho_res, theta_res)

        for num_lines in num_lines_list:
            line_rhos, line_thetas = hough_lines_peak(H, rho_res, theta_res, num_lines)
            
            image_with_lines = draw_lines(image, line_rhos, line_thetas)

            fname = (f"./results/my_image_01"
                     f"_sigma={sigma}"
                     f"_thr={threshold}"
                     f"_rho={rho_res}"
                     f"_n={num_lines}.png")
            image_with_lines.save(fname)

            count += 1
            print(f"[{count}/{total}] {fname}")

[1/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=1.0_n=15.png
[2/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=1.0_n=20.png
[3/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=1.0_n=30.png
[4/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=1.5_n=15.png
[5/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=1.5_n=20.png
[6/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=1.5_n=30.png
[7/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=2.3_n=15.png
[8/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=2.3_n=20.png
[9/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=2.3_n=30.png
[10/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=3.0_n=15.png
[11/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=3.0_n=20.png
[12/300] ./results/my_image_01_sigma=0.5_thr=0.02_rho=3.0_n=30.png
[13/300] ./results/my_image_01_sigma=0.5_thr=0.04_rho=1.0_n=15.png
[14/300] ./results/my_image_01_sigma=0.5_thr=0.04_rho=1.0_n=20.png
[15/300] ./results/my_image_01_sigma=0.5_thr=0.04_rho=1.0_n=30.png
[16/

In [34]:
import os
from itertools import product

######## Parameters ########
# default #
# sigma=2 # 얼마나 흐릿하게 만드는가? 커질수록 노이즈 제거 강화
# threshold=0.08 # 최소 강도
# rho_res=2.3 # 정밀도; 작을수록 정밀
# theta_res=math.pi/180
# num_lines=20 # 검출할 직선 개수

sigma_list     = [0.5, 1, 2, 3, 5]
threshold_list = [0.02, 0.04, 0.08, 0.1, 0.2]
rho_res_list   = [1.0, 1.5, 2.3, 3.0]
num_lines_list = [15, 20, 30]
theta_res      = math.pi / 180
######## Parameters ########

image = Image.open("./my_image_03.jpg")
image_grey = np.array(image.convert("L")) / 255.

total = len(sigma_list) * len(threshold_list) * len(rho_res_list) * len(num_lines_list)
count = 0

for sigma in sigma_list:

    edge_image, _, __, ___ = edge_detection(image_grey, sigma)

    for threshold, rho_res in product(threshold_list, rho_res_list):
        H = hough_lines(edge_image, threshold, rho_res, theta_res)

        for num_lines in num_lines_list:
            line_rhos, line_thetas = hough_lines_peak(H, rho_res, theta_res, num_lines)
            
            image_with_lines = draw_lines(image, line_rhos, line_thetas)

            fname = (f"./results/my_image_03"
                     f"_sigma={sigma}"
                     f"_thr={threshold}"
                     f"_rho={rho_res}"
                     f"_n={num_lines}.png")
            image_with_lines.save(fname)

            count += 1
            print(f"[{count}/{total}] {fname}")

[1/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=1.0_n=15.png
[2/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=1.0_n=20.png
[3/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=1.0_n=30.png
[4/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=1.5_n=15.png
[5/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=1.5_n=20.png
[6/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=1.5_n=30.png
[7/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=2.3_n=15.png
[8/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=2.3_n=20.png
[9/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=2.3_n=30.png
[10/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=3.0_n=15.png
[11/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=3.0_n=20.png
[12/300] ./results/my_image_03_sigma=0.5_thr=0.02_rho=3.0_n=30.png
[13/300] ./results/my_image_03_sigma=0.5_thr=0.04_rho=1.0_n=15.png
[14/300] ./results/my_image_03_sigma=0.5_thr=0.04_rho=1.0_n=20.png
[15/300] ./results/my_image_03_sigma=0.5_thr=0.04_rho=1.0_n=30.png
[16/

In [35]:
######## Parameters ########

# default #
# sigma=2 
# threshold=0.1
# a_res=2.0
# b_res=2.0
# r_res=4.0
# radius_range=(120, 180)
# num_circles=3

sigma=2
threshold=0.1
a_res=2.0
b_res=2.0
r_res=4.0
radius_range=(120, 180)
num_circles=3
######## Parameters ########

image = Image.open("./my_image_04.jpg")
image_grey = np.array(image.convert("L"))
image_grey = image_grey / 255.

edge_image, _, __, ___ = edge_detection(image_grey, sigma)
H = hough_circles(edge_image, threshold, radius_range, a_res, b_res, r_res)
circle_as, circle_bs, circle_rs = hough_circles_peak(H, radius_range, a_res, b_res, r_res, num_circles)
image_with_circles = draw_circles(image, circle_as, circle_bs, circle_rs)

image_with_circles.save("./my_image_04_output.jpg")